In [1]:
import sys
import os

import streamlit as st
from core.utils import Sermon, Person, get_short_info
from collections import Counter

import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx

import plotly.graph_objects as go
import pandas as pd

import json
import re

from rapidfuzz import fuzz

In [15]:
from nltk.corpus import stopwords
german_stop_words = set(stopwords.words('german'))
em_stopwords = []
for stopword in german_stop_words:
    if "i" in stopword:
        em_stopwords.append(stopword)
        em_stopwords.append(stopword.replace("i", "j"))
        em_stopwords.append(stopword.replace("i", "y"))
    elif "u" in stopword:
        em_stopwords.append(stopword)
        em_stopwords.append(stopword.replace("u", "v"))
    else:
        em_stopwords.append(stopword)

em_stopwords = set(em_stopwords)

In [3]:
sermon = Sermon("E000036")

In [8]:
sermon.chunked[8]

[{'words': ['herrn', 'carl', 'christian', 'lange,'],
  'types': ['', '', '', ''],
  'references': [[], [], [], []]},
 {'words': ['hochverdienten',
   'herrn',
   'rathsverwandten',
   'und',
   'zu',
   'der',
   'zeit',
   'hochpreißlich',
   'gewesenen',
   'hospitalsherrn,'],
  'types': ['', '', '', '', '', '', '', '', '', ''],
  'references': [[], [], [], [], [], [], [], [], [], []]},
 {'words': ['seiner',
   'hohen',
   'und',
   'mit',
   'tiefstem',
   'respect',
   'zu',
   'verehrenden',
   'obrigkeit'],
  'types': ['', '', '', '', '', '', '', '', ''],
  'references': [[], [], [], [], [], [], [], [], []]}]

In [10]:
words = sermon.chunked[8][1]["words"]

words

['hochverdienten',
 'herrn',
 'rathsverwandten',
 'und',
 'zu',
 'der',
 'zeit',
 'hochpreißlich',
 'gewesenen',
 'hospitalsherrn,']

In [11]:
[word for word in words if word not in em_stopwords]

['hochverdienten',
 'herrn',
 'rathsverwandten',
 'zeit',
 'hochpreißlich',
 'gewesenen',
 'hospitalsherrn,']

In [16]:
verse = "hr Chriften gut / Habt frischen Muth/ Den Raub habn wir bekommen/ Gerechtigkeit Ist unsre Bent/Wir sind der Furcht entnommen/ Hier ist die Beut/Gerechtigkeit/ Nun sind wir Gottes Kinder/ Drum singn wir all Mit Freudenschall: Danck sey dem Uberwinder"

In [17]:
[word.lower() for word in verse.split() if word.lower() not in em_stopwords]

['hr',
 'chriften',
 'gut',
 '/',
 'habt',
 'frischen',
 'muth/',
 'raub',
 'habn',
 'bekommen/',
 'gerechtigkeit',
 'unsre',
 'bent/wir',
 'furcht',
 'entnommen/',
 'beut/gerechtigkeit/',
 'gottes',
 'kinder/',
 'drum',
 'singn',
 'all',
 'freudenschall:',
 'danck',
 'sey',
 'uberwinder']

In [27]:
def flatten(xss):
    return [x for xs in xss for x in xs]

In [125]:
def is_id(value):
    pattern = re.compile(r'E[01][0-9]{5}')
    if re.match(pattern, value):
        return True
    else:
        return False

In [28]:
def create_quote_dist_chart(ids: list, type: str, searchkey: str="") -> go.Figure:
    type_dict = {
        "orgelpredigt": "Orgelpredigtzitate",
        "musikwerk": "Liedzitate",
        "quelle": "Literaturzitate",
    }
    if type not in type_dict.keys():
        occ_fig = go.Figure()
        occ_fig.update_layout(title_text="Type not recognised!")
        return occ_fig
    
    else:
        chunked_text = [0]*100
        thumbnails = [""]*100

        for id in ids:
            sermon = Sermon(id)

            dec = int(len(sermon.words) / 99)
            overhang = len(sermon.words) % dec

            for i, j in zip(range(0, len(sermon.words), dec), range(0, 100)):
                if searchkey != "":
                    keys_unique = sermon.reference[i:i+dec]
                    keys_str = " ".join(flatten(keys_unique))
                    if searchkey in keys_str:
                        hit_test = 1
                        hit = f"{sermon.kurztitel}"
                    else:
                        hit_test = 0
                        hit = ""
                else:
                    types_unique = list(set(sermon.word_types[i:i+dec]))
                    types_str = " ".join([x for x in types_unique if isinstance(x, str)])
                    if type in types_str:
                        hit_test = 1
                        hit = f"{sermon.kurztitel}<br>"
                    else:
                        hit_test = 0
                        hit = ""
                
                chunked_text[j] = chunked_text[j] + hit_test
                thumbnails[j] = thumbnails[j] + hit
            if searchkey != "":
                last_keys_unique = sermon.reference[-overhang:]
                last_keys_str = " ".join(flatten(last_keys_unique))
                if searchkey in last_keys_str:
                    last_hit_test = 1
                    last_hit = f"{sermon.kurztitel}<br>"
                else:
                    last_hit_test = 0
                    last_hit = ""
            else:
                last_types_unique = list(set(sermon.word_types[-overhang:]))
                last_types_str = " ".join([x for x in last_types_unique if isinstance(x, str)])
                if type in last_types_str:
                    last_hit_test = 1
                    last_hit = f"{sermon.kurztitel}<br>"
                else:
                    last_hit_test = 0
                    last_hit = ""
            
            #chunked_text[-1] = chunked_text[-1] + last_orgelpredigt_test
            #thumbnails[-1] = thumbnails[-1] + last_hit

        occ_fig = go.Figure()

        for i in range(0, len(chunked_text)):
            hovertext = f'{chunked_text[i]} {type_dict[type]} im {i+1}%'
            if thumbnails[i] != "":
                    hovertext += f"<br>{thumbnails[i]}"

            gradient = chunked_text[i] * 15
            color = f'rgb({max(250-gradient, 0)},{max(250-gradient, 0)},{max(250-gradient, 0)})'
            occ_fig.add_trace(go.Bar(
                x = [f"{type_dict[type]} je Predigtprozent"],
                y = [100],
                marker_color = color,
                hovertext = hovertext
            ))

        occ_fig.update_layout(width=1500,height=500, showlegend=False)
        print(hit)
        return occ_fig

def group_sermons_in_years(data, interval: int) -> list:
    chunked_sermons = []
    start_year = 1600
    end_year = 1800
    yearfinder = re.compile(r'[0-9]{4}')
    for i in range(start_year, end_year, interval):
        sermons = []
        for id, info in data.items():
            year = int(re.findall(yearfinder, info['year'])[0])
            if year > i and year < i + interval:
                sermons.append(id)
        chunked_sermons.append(sermons)

    return chunked_sermons

In [29]:
# Get the list of all files in a directory
with open("predigten_übersicht.json", "r", encoding="utf-8") as file: 
    data = json.load(file)

# Ensure all entries have a 'year' key
cleaned = {k: v for k, v in data.items() if 'year' in v}

year_finder = re.compile(r'[0-9]{4}')

for k, v in data.items():
    year = re.findall(year_finder, v['year'])[0]
    if year:
        v['year'] = year
    else:
        v['year'] = '[s.a.]'

# Convert to nested list and sort by year
relevant_sermons = sorted(
    [[key, value['title'], int(value['year'])] for key, value in cleaned.items()],
    key=lambda x: x[2]
)

ids = [i[0] for i in relevant_sermons]

In [31]:
praetorius = "E080223"
in_dulci = "E100017"
dieterich = "E000003"
morgenstern = "E100022"

In [32]:
quote_time_dist = "gesamter_zeitraum"
quote_type = "musikwerk"
searchkey = morgenstern

In [33]:
if quote_time_dist == "50-Jahr-Intervalle":
    sermons_grouped_50 = group_sermons_in_years(data, 50)
    figs_50 = []
    for i in range(len(sermons_grouped_50)):
        figs_50.append(create_quote_dist_chart(sermons_grouped_50[i], quote_type,searchkey=searchkey))
    
    # Create subplots
    fig = make_subplots(rows=len(figs_50), 
                        cols=1, 
                        subplot_titles=[f"Verteilung in Predigten zwischen {1600 + (i*50)} und {1600+(i*50)+50} ({len(sermons_grouped_50[i])} Predigten)" for i in range(len(figs_50))])

    # Add traces from each figure to the subplots
    for i, fig_item in enumerate(figs_50):
        for trace in fig_item.data:
            fig.add_trace(trace, row=i+1, col=1)

    # Update layout
    fig.update_layout(height=1200, width=1000, showlegend = False)
    fig.update_layout(title_text="Accumulierte Verteilung von Zitaten in 50-Jahr Intervallen")

elif quote_time_dist == "25-Jahr-Intervalle":
    sermons_grouped_25 = group_sermons_in_years(data, 25)
    figs_25 = []
    for i in range(len(sermons_grouped_25)):
        figs_25.append(create_quote_dist_chart(sermons_grouped_25[i], quote_type,searchkey=searchkey))
    
    # Create subplots
    fig = make_subplots(rows=len(figs_25), 
                        cols=1, 
                        subplot_titles=[f"Verteilung in Predigten zwischen {1600 + (i*25)} und {1600+(i*25)+25} ({len(sermons_grouped_25[i])} Predigten)" for i in range(len(figs_25))])

    # Add traces from each figure to the subplots
    for i, fig_item in enumerate(figs_25):
        for trace in fig_item.data:
            fig.add_trace(trace, row=i+1, col=1)

    # Update layout
    fig.update_layout(height=1200, width=1000, showlegend = False)
    fig.update_layout(title_text="Accumulierte Verteilung von Zitaten in 25-Jahr Intervallen")

else:
    fig = create_quote_dist_chart(ids, quote_type, searchkey=searchkey)

fig.show()

In [34]:
sermon = Sermon("E000036")
quoted_item = "E100022"
passages = []
for i in range(len(sermon.chunked)):
    for j in range(len(sermon.chunked[i])):
        words = sermon.chunked[i][j]["words"]
        types = sermon.chunked[i][j]["types"]
        refs = sermon.chunked[i][j]["references"]
        test_refs = []
        for nr, ref in enumerate(refs):
            if quoted_item in ref:
                test_refs.append(nr)
        if len(test_refs):
            passages.append(" ".join(words))


In [35]:
passages

['zwingt die säiten in cithara,',
 'und laßt die süsse musica gantz freudenreich erschallen,',
 'daß ich möge mit jesulein,',
 'dem wunderschönen bräutgam mein,',
 'jn steter liebe wallen.',
 'singet, springet, jubiliret, triumphiret,',
 'danckt dem herren, groß ist der könig der ehren!',
 'singet, springet, jubiliret, triumphiret,',
 'danckt dem herren, groß ist der könig der ehren.']

In [ ]:
#def get_neighbouring_quotes(sermon_chunked, par_nr, sent_nr, id):
#    par = sermon_chunked[par_nr]


In [116]:
def get_shared_quotes(quoted_item: str, ids: list):
    shared_quotes = []
    for id in ids:
        sermon = Sermon(id)
        for par in sermon.chunked:
            refs = flatten(flatten([x["references"] for x in par]))
            if quoted_item in refs:
                shared_quotes.extend(refs)
    return [x for x in shared_quotes if x != quoted_item]

In [122]:
"Sir_43".split("-")

['Sir_43']

In [135]:
morgenstern_adjacent_quotes = get_shared_quotes(morgenstern, ids)
morgenstern_adjacent_quotes = [x.split("-")[0] for x in morgenstern_adjacent_quotes]
morgenstern_adjacent_quotes = [get_short_info(x) if is_id(x) else x for x in morgenstern_adjacent_quotes]
morgenstern_adjacent_quotes = [x.split(":")[1] if len(x.split(":")) > 1 else x for x in morgenstern_adjacent_quotes]

In [139]:
counts = Counter(morgenstern_adjacent_quotes)
top20 = counts.most_common(20)

# Sort by frequency (descending)
sorted_items = counts.most_common()
labels = [item[0] for item in top20]
values = [item[1] for item in top20]

# Plot with Plotly
fig = px.bar(
    x=labels, 
    y=values, 
    labels={'x': 'Quelle', 'y': 'Häufigkeit'},
    title="20 am häufigsten mit 'Wie schön leuchtet der Morgenstern' im selben Paragraph erscheinenden Quellen/Lieder"
)
fig.show()

In [142]:
def get_quoted_passages(quoted_item: str, ids: list):
    # get passages from quoted item in every sermon
    passages = []
    for id in ids:
        sermon = Sermon(id)
        for i in range(len(sermon.chunked)):
            for j in range(len(sermon.chunked[i])):
                words = sermon.chunked[i][j]["words"]
                types = sermon.chunked[i][j]["types"]
                refs = sermon.chunked[i][j]["references"]
                test_refs = []
                for nr, ref in enumerate(refs):
                    if quoted_item in ref:
                        test_refs.append(nr)
                if len(test_refs):
                    if j > 0:
                        words_before = " ".join(sermon.chunked[i][j-1]["words"])
                    else:
                        words_before = " ".join(sermon.chunked[i-1][-1]["words"])
                    if j < len(sermon.chunked[i])-1:
                        words_after = " ".join(sermon.chunked[i][j+1]["words"])
                    else:
                        words_after = " ".join(sermon.chunked[i+1][0]["words"])
                    passages.append([id, i, j, " ".join(words), words_before, words_after])
    
    return passages

passages = get_quoted_passages(morgenstern, ids)

In [11]:
dieterich_text = Sermon("E000003").chunked

In [15]:
dieterich_text[0][0]

{'words': ['vlmische',
  'orgel',
  'predigt/',
  'darinn',
  'von',
  'der',
  'jnstrumentalmusic',
  'inns',
  'gemein/'],
 'types': ['', '', '', '', '', '', '', '', ''],
 'references': [[], [], [], [], [], [], [], [], []]}

In [16]:
dieterich_verses = []
for par in dieterich_text:
    for sent in par:
        dieterich_verses.append(" ".join(sent['words']))
dieterich_verses

['vlmische orgel predigt/ darinn von der jnstrumentalmusic inns gemein/',
 'sonderlich aber von dero orgeln erfindung vnd gebrauch/',
 'in der kirchen gottes/',
 'von anfang der welt biß hieher,',
 'kürtzlich discurriret, zugleich auch die schöne herrliche vlmer orgel beschrieben wirdt.',
 'gehalten zu vlm im münster/',
 'an dessen kirchweyhtag/ den 1.',
 'augusti dieses 1624. jahrs/',
 'vnd auff begehren in offenen truck geben/',
 'durch cunrad dieterich/ der heiligen schrifft doctorn/',
 'dero vlmischen kirchen superintendenten.',
 'gedruckt zu vlm in der mederischen truckerey.',
 'm. dc. xxiv',
 'vakat',
 'ehrnvöster/ vornehmer/ insonders geliebter herr vnd freundt/',
 'demnach ich ohnlangst bey vnserer nechstverwichnen kirchwey/',
 'ein christliche einfältige sermon/',
 'von der jnstrumentalmusic vnd orgeln gehalten/',
 'darinn ich von dem ersten brauch der musicalischen jnstrumenten/',
 'beym gottesdienst ins gemein/',
 'besonders aber deren orgeln/',
 'einen kurtz durchgehenden d

In [144]:
results = pd.DataFrame(passages, columns=['sermon', 'par', 'sent', 'verse', 'text_before', 'text_after'])

In [39]:
liedtext_morgenstern = [
"Wie schön leuchtet der Morgenstern /",
"Voll Gnad vnd Warheit von dem HERRN /",
"Die süsse Wurtzel Jesse?",
"Du Sohn Dauid auß Jacobs Stamm /",
"Mein König vnd mein Bräutigam /",
"Hast mir mein Hertz besessen /",
"Lieblich freundtlich Schön vnd herrlich /",
"Groß vnd ehrlich Reich von Gaben /",
"Hoch vnd sehr prächtig erhaben.",
"Ey mein Perle / du werthe Kron /",
"Wahr Gottes vnd Marien Sohn /",
"Ein hochgeborner König /",
"Mein Hertz heißt dich ein lilium,",
"Dein süsses Euangelium,",
"Jst lauter Milch vnd Honig /",
"Ey mein Blümlein Hosianna /",
"Himmlisch Manna Das wir essen /",
"Deiner kan ich nicht vergessen.",
"Geuß sehr tieff in mein Hertz hineyn /",
"Du heller Jaspis vnd Rubin /",
"Die Flamme deiner Liebe.",
"Vnd erfreuw mich / daß ich doch bleib",
"An deinem außerwehlten Leib",
"Ein lebendige Rippe /",
"Nach dir ist mir /",
"Gratiosa cœli rosa Kranck vnd glümmet",
"Mein Hertz / durch Liebe verwundet.",
"Von Gott kompt mir ein Frewdenschein /",
"Wenn du mit deinen Eugelein /",
"Mich freundtlich thust anblicken /",
"O HERR Jesu mein trawtes Gut /",
"Dein Wort / dein Geist / dein Leib vnd Blut /",
"Mich innerlich erquicken. Nimm mich /",
"freundtlich Jn dein Arme /",
"Daß ich warme Werd von Gnaden /",
"Auff dein Wort komm ich geladen.",
"HERR Gott Vatter / mein starcker Heldt /",
"Du hast mich ewig / für der Welt /",
"In deinem Sohn geliebet /",
"Dein Sohn hat mich jhm selbst vertrawt /",
"Er ist mein Schatz / ich bin sein Braut /",
"Sehr hoch in jhm erfreuwet.",
"Eya Eya Himmlisch Leben /",
"wirdt er geben Mir dort oben /",
"Ewig soll mein Hertz jhn loben.",
"Zwingt die Sayten in Cythara.",
"Vnd laßt die süsse Musica,",
"Gantz frewdenreich erschallen:",
"Daß ich möge mit Jesulein /",
"Dem wunder schönen Bräutgam mein /",
"In stäter Liebe wallen.",
"Singet springet Jubilieret triumphieret /",
"Danckt dem HERREN Groß ist der König der Ehren.",
"Wie bin ich doch so hertzlich fro /",
"Daß mein Schatz ist das A vnd O /",
"Der Anfang / vnd das Ende:",
"Er wirdt mich doch zu seinem Preyß /",
"Auffnemmen in das Paradeiß /",
"Deß klopff ich in die Hände.",
"Amen Amen Komm du schone",
"FrewdenKrone Bleib du nicht lange /",
"Deiner wart ich mit Verlangen."
]

In [145]:
results

,sermon,par,sent,verse,text_before,text_after
0,E000091,60,0,ey/ mein perle du werthe cron/,was ich gesucht bin ich gewehrt.,wahr gottes und marien sohn/
1,E000091,60,1,wahr gottes und marien sohn/,ey/ mein perle du werthe cron/,ein hochgeborner könig/ mein hertz heisst dich...
2,E000091,60,2,ein hochgeborner könig/ mein hertz heisst dich...,wahr gottes und marien sohn/,dein süsses evangelium/ ist lauter milch und h...
3,E000091,60,3,dein süsses evangelium/ ist lauter milch und h...,ein hochgeborner könig/ mein hertz heisst dich...,ey mein/ blümlein/ hosianna/
4,E000091,60,4,ey mein/ blümlein/ hosianna/,dein süsses evangelium/ ist lauter milch und h...,himlisch manna/ daß wir essen/
5,E000091,60,5,himlisch manna/ daß wir essen/,ey mein/ blümlein/ hosianna/,deiner kan ich nicht vergessen.
6,E000091,60,6,deiner kan ich nicht vergessen.,himlisch manna/ daß wir essen/,zwingt die seiten in cythara/
7,E000091,61,0,zwingt die seiten in cythara/,deiner kan ich nicht vergessen.,vnd lasst die süsse musica/
8,E000091,61,1,vnd lasst die süsse musica/,zwingt die seiten in cythara/,ganz freudenreich erschallen/ daß ich möge mit...
9,E000091,61,2,ganz freudenreich erschallen/ daß ich möge mit...,vnd lasst die süsse musica/,den allerhöchsten bräutgam mein/


In [ ]:
def get_full_quotes(df):
    # Ensure proper order
    df = df.sort_values(["sermon","par","sent"]).reset_index(drop=True)

    # Detect breaks in consecutiveness *per sermon and par*
    df["break"] = (
        (df["sent"] != df["sent"].shift(1) + 1) | 
        (df["par"] != df["par"].shift(1)) |
        (df["sermon"] != df["sermon"].shift(1))
    )

    # Create group id
    df["grp"] = df["break"].cumsum()

    # Group and concatenate
    # aggregate while keeping sermon/par in output
    out = (
        df.groupby(["sermon","par","grp"], as_index=False)
        .agg(
            sent_min = ("sent","min"),
            sent_max = ("sent","max"),
            verse = ("verse", " ".join)
        )
    )

    # optional: pretty sent_range and drop grp
    out["sent_range"] = out.apply(
        lambda r: str(r["sent_min"]) if r["sent_min"]==r["sent_max"] else f"{r['sent_min']}-{r['sent_max']}",
        axis=1
    )
    out = out.drop(columns="grp")
    return out


In [162]:
def get_sent_before(id, par, sent):
    sermon = Sermon(id)
    if sent != 0:
        return " ".join(sermon.chunked[par][sent-1]["words"])
    else:
        return ""
def get_sent_after(id, par, sent):
    sermon = Sermon(id)
    if sent+1 != len(sermon.chunked[par]):
        return " ".join(sermon.chunked[par][sent+1]["words"])
    else:
        return ""

In [ ]:
full_quotes = get_full_quotes(results)
full_quotes["before"] = full_quotes.apply(
    lambda r: get_sent_before(r["sermon"], r["par"], r["sent_min"]), axis=1
)
full_quotes["after"] = full_quotes.apply(
    lambda r: get_sent_after(r["sermon"], r["par"], r["sent_max"]), axis=1
)
full_quotes

,sermon,par,sent_min,sent_max,verse,sent_range,before,after
0,E000036,30,0,6,"zwingt die säiten in cithara, und laßt die süs...",0-6,,
1,E000036,75,0,1,"singet, springet, jubiliret, triumphiret, danc...",0-1,,
2,E000042,34,0,5,"zwingt die saiten in cythara, und lasst die sü...",0-5,,amen!
3,E000052,20,18,18,zwingt die säyten in cythara etc.,18,singet mit einstimmen unserer lobschallenden n...,und bethet darauf ein gläubiges und andächtige...
4,E000052,25,31,33,"zwingt die säyten in cithara, und last die süs...",31-33,amabimus &amp; laudabimus wir werden gott lieb...,"nimmermehr kan eine orgel so viel züge haben,"
5,E000056,6,2,2,das eintzige gesäzz: zwingt die saiten in cith...,2,mit einander werden gesungen und geklungen haben/,
6,E000061,13,0,0,herr gott vater mein starcker held etc.,0,,den 5. 6. und 7.
7,E000061,13,3,3,wie schön leuchtet der morgenstern.,3,v. aus dem liede:,
8,E000068,45,0,2,"zwingt die seiten in cithara, zwingt die pfeif...",0-2,,
9,E000070,80,0,3,"so zwingt die säiten in cithara, und lasst die...",0-3,,


In [196]:
relevant_sermons

[['E000001', 'Christliche Predigt (Tübingen 1602)', 1602],
 ['E000002', 'Musica instrumentalis (Meißen 1605)', 1605],
 ['E000029', 'Christliche Predigt (Tübingen 1606)', 1606],
 ['E000030', 'Elogium Organi Musici (Altenburg 1610)', 1610],
 ['E000099', 'Corona Templi (Nürnberg 1621)', 1621],
 ['E000003', 'Vlmische Orgel Predigt (Ulm 1624)', 1624],
 ['E000098', 'Musica ecclesiastica (Stettin 1628)', 1628],
 ['E000096', 'Kostbare Bosische Orgel (Zwickau 1647)', 1647],
 ['E000095',
  'Längst=gewüntzschte Mittweidische Orgel=Freude (Dresden 1648)',
  1648],
 ['E000092', 'Organologismos (Dresden 1651)', 1651],
 ['E000091', 'Stolpenische Ehren-Crone (Dresden 1652)', 1652],
 ['E000090', 'Organolustria Evangelico-Stambachiana (Hof 1660)', 1660],
 ['E000089', 'Encoenia HierOrganica (Halle 1664)', 1664],
 ['E000086', 'Orgel=Predigt (Arnstadt 1666)', 1666],
 ['E000085', 'Das fröliche Hallelujah (Halle 1667)', 1667],
 ['E000083', 'Das Gott=Lob=Schallende Hosianna (Leipzig 1671)', 1671],
 ['E000082'

In [164]:
def create_html(df):
    df["formatted"] = df.apply(
    lambda r: f'{r['before']} <span class="actual_quote">{r['verse']}</span>{r['after']}', axis=1
)
    sermon_dict = df.groupby("sermon")["formatted"].agg(list).to_dict()
    return sermon_dict

In [175]:
x = [{"key1": "val1"}, {"key2": "val2"}]
all_keys = []
for xx in x:
    y = [a for a,b in xx.items()]
    all_keys.extend(y)
all_keys

['key1', 'key2']

In [177]:
for elem in x:
    if "key1" in elem.keys():
        print(elem["key1"])

val1


In [179]:
[b["key1"] for b in x if "key1" in b.keys()]

['val1']

In [ ]:
with open("predigten_übersicht.json", "r", encoding="utf-8") as file: 
    data = json.load(file)

# Ensure all entries have a 'year' key
cleaned = {k: v for k, v in data.items() if 'year' in v}

year_finder = re.compile(r'[0-9]{4}')

for k, v in data.items():
    year = re.findall(year_finder, v['year'])[0]
    if year:
        v['year'] = year
    else:
        v['year'] = '[s.a.]'

# Convert to nested list and sort by year
relevant_sermons = sorted(
    [[key, value['title'], int(value['year'])] for key, value in cleaned.items()],
    key=lambda x: x[2]
)

ids = [x[0] for x in relevant_sermons]

source_options = []
for id in ids:
    sermon = Sermon(id)
    source_options.extend(set(flatten(sermon.reference)))

source_options = [x for x in source_options if is_id(x)]

id_counts = Counter(source_options)

In [194]:
# Sort IDs by frequency descending
sorted_ids = [[item[0], item[1]] for item in id_counts.most_common()]
options = [{f'{get_short_info(i[0])} ({i[1]} Verweise)' :i[0]} for i in sorted_ids]
keys = []
for x in options:
    key = [key for key, val in x.items()]
    keys.extend(key)

In [195]:
keys

['Praetorius, Michael: Syntagmatis Musici Michaelis Praetorii C. Tomus Secundus De Organographia (1619) (22 Verweise)',
 'anonym: In dulci jubilo (16 Verweise)',
 'Gratianus de Clusio: Decretum Gratiani emendatum & notationibus illustratum, una cum glossis (1604) (14 Verweise)',
 'Nicolai, Philipp: Wie schön leuchtet der Morgenstern (13 Verweise)',
 'Augustinus, Aurelius ; Thimme, Wilhelm (Übers.): Confessiones. Bekenntnisse (2004) (12 Verweise)',
 'Anonym: Ach Gott wie manches Herzeleid (12 Verweise)',
 'Conrad Dieterich, 1575/01/09 (Gemünden (Wohra))-1639/03/22 (Ulm): Vlmische Orgel Predigt (Ulm 1624) (11 Verweise)',
 'Zwinger, Theodor: Theatrvm Hvmanae Vitae Theodori Zuingeri Bas[isliensis] Tertiatione ([1586]) (8 Verweise)',
 'N.N.: Herr Gott, dich loben wir (8 Verweise)',
 'Crüger, Johann: Sei Lob und Ehr dem höchsten Gut (8 Verweise)',
 'Luther, Martin: D. Martin Luthers Werke (1969) (7 Verweise)',
 'Augustinus, Aurelius ; Gori, Franco (Hrsg.): Enarrationes in psalmos\xa0101–150 

In [165]:
create_html(full_quotes)

{'E000036': [' <span class="actual_quote">zwingt die säiten in cithara, und laßt die süsse musica gantz freudenreich erschallen, daß ich möge mit jesulein, dem wunderschönen bräutgam mein, jn steter liebe wallen. singet, springet, jubiliret, triumphiret, danckt dem herren, groß ist der könig der ehren!</span>',
  ' <span class="actual_quote">singet, springet, jubiliret, triumphiret, danckt dem herren, groß ist der könig der ehren.</span>'],
 'E000042': [' <span class="actual_quote">zwingt die saiten in cythara, und lasst die süsse musica, ganz freudenreich erschallen, daß ich möge mit jesulein dem wunderschönen bräutgam mein, jn steter liebe wallen! singet, springet, jubiliret, triumphiret, danckt dem herren, groß ist der könig der ehren.</span>amen!'],
 'E000052': ['singet mit einstimmen unserer lobschallenden neuen orgel: <span class="actual_quote">zwingt die säyten in cythara etc.</span>und bethet darauf ein gläubiges und andächtiges vater unser',
  'amabimus &amp; laudabimus wir we

In [74]:
def get_closest_verse(sermon_line: str, source: list) -> list:
    best_match = ""
    best_score = 0
    for verse in source:
        sim_score = fuzz.ratio(sermon_line, verse.lower())
        if sim_score > best_score:
            best_score = sim_score
            best_match = verse
    
    return [best_match, best_score, source.index(best_match)]

In [75]:
results[["best_match", "sim_score", "line"]] = results["verse"].apply(lambda x: pd.Series(get_closest_verse(x, dieterich_verses)))

In [76]:
results

,sermon,verse,text_before,text_after,best_match,sim_score,line
0,E000091,mit hertzlicher dancksagung; denn nechst reine...,"1. deo pro organo gratias agendo,",wie auch rechter ausspendung der heiligen sacr...,mit welchen man den herrn in seinem heyligtumb...,52.941176,276
1,E000091,wie auch rechter ausspendung der heiligen sacr...,mit hertzlicher dancksagung; denn nechst reine...,"und einer guten vocal-music,",vnnd rechtem gebrauch/ der hochwürdigen sacram...,62.264151,1227
2,E000091,"und einer guten vocal-music,",wie auch rechter ausspendung der heiligen sacr...,sind die orgeln in kirchen eine sonderliche zi...,weder orgel noch music:,54.901961,1308
3,E000091,sind die orgeln in kirchen eine sonderliche zi...,"und einer guten vocal-music,",als dadurch nicht nur der öffentliche gottesdi...,vnd denen orgeln/ in der kirchen gottes/,59.340659,908
4,E000091,als dadurch nicht nur der öffentliche gottesdi...,sind die orgeln in kirchen eine sonderliche zi...,und ansehnlich gemachet wird/,sondern auch der offene gottesdienst in der ki...,64.285714,1233
...,...,...,...,...,...,...,...
318,E000074,oder wasserorgeln/ so durch wasser gestimmet w...,man hat auch hydraulica gehabt/,allein constantini pipino verehrte orgelpfeiff...,so durch wasser gestimmet worden/,75.294118,702
319,E000074,von welcher zeit an die orgeln in teutschland ...,ebenfalls von dem griechischen kayser constant...,daß fast keine vornehme kirch in städten/,die orgel inn den teutschen kirchen allgemach ...,63.492063,733
320,E000074,daß fast keine vornehme kirch in städten/,von welcher zeit an die orgeln in teutschland ...,flecken/ dörffern und clöstern gewesen/,daß fast kein vornehme kirch in stätten/,96.296296,734
321,E000074,flecken/ dörffern und clöstern gewesen/,daß fast keine vornehme kirch in städten/,welche nicht ein orgelwerck gehabt hätte.,welches ein recht gewesen/,55.384615,378


In [77]:
quoting_sermons = results['sermon'].values.tolist()

In [78]:
x = results.loc[results["best_match"] == "Singet springet Jubilieret triumphieret /"].values.tolist()[0]

IndexError: list index out of range

In [79]:
lines_with_data = []
for line in dieterich_verses:
    row_list = results.loc[results["best_match"] == line].values.tolist()
    lines_with_data.append({"name": line, "value":row_list})

lines_with_data

[{'name': 'vlmische orgel predigt/ darinn von der jnstrumentalmusic inns gemein/',
  'value': []},
 {'name': 'sonderlich aber von dero orgeln erfindung vnd gebrauch/',
  'value': []},
 {'name': 'in der kirchen gottes/',
  'value': [['E000090',
    'sind das wort gottes/',
    'die bälge/ dadurch solcher wind getrieben/',
    'das clavier und pedal ist unser hertz/',
    'in der kirchen gottes/',
    65.11627906976744,
    2]]},
 {'name': 'von anfang der welt biß hieher,', 'value': []},
 {'name': 'kürtzlich discurriret, zugleich auch die schöne herrliche vlmer orgel beschrieben wirdt.',
  'value': []},
 {'name': 'gehalten zu vlm im münster/',
  'value': [['E000073',
    'zu ulm in münster gethan.',
    'dergleichen sie anno 1531.',
    'auf daß sie aber die orgelwerck mögen verdächtig und verhast machen/',
    'gehalten zu vlm im münster/',
    57.692307692307686,
    5]]},
 {'name': 'an dessen kirchweyhtag/ den 1.', 'value': []},
 {'name': 'augusti dieses 1624. jahrs/', 'value': []},
 

In [80]:
lines_cleaned = []
for line in lines_with_data:
    line_info = {}
    line_info["text"] = line["name"]
    line_info["quotes"] = len(line["value"])
    line_info["details"] = line["value"]
    lines_cleaned.append(line_info)

In [81]:
len(lines_cleaned)

1607

In [82]:
percent = len(lines_cleaned)/100

In [197]:
cleaned

{'E000001': {'title': 'Christliche Predigt (Tübingen 1602)',
  'year': '1602',
  'length': 6560,
  'worte_bibel': 380,
  'worte_quellen': 464,
  'worte_orgelpredigt': 0,
  'worte_musikwerk': 0,
  'zitierte_musikwerke': 0,
  'bibelstelle': 'Ps 150',
  'verlagsort': 'Tübingen',
  'einweihungsort': 'Memmingen'},
 'E000002': {'title': 'Musica instrumentalis (Meißen 1605)',
  'year': '1605',
  'length': 9118,
  'worte_bibel': 1825,
  'worte_quellen': 5,
  'worte_orgelpredigt': 0,
  'worte_musikwerk': 0,
  'zitierte_musikwerke': 0,
  'bibelstelle': 'Ps 69,3',
  'verlagsort': 'Leipzig',
  'einweihungsort': 'Meißen'},
 'E000003': {'title': 'Vlmische Orgel Predigt (Ulm 1624)',
  'year': '1624',
  'length': 10834,
  'worte_bibel': 498,
  'worte_quellen': 1464,
  'worte_orgelpredigt': 0,
  'worte_musikwerk': 0,
  'zitierte_musikwerke': 0,
  'bibelstelle': 'Ps 150, 1-6',
  'verlagsort': 'Ulm',
  'einweihungsort': 'Ulm, Münster ; E030003'},
 'E000007': {'title': 'Predigt bey der feyerlichen Einweih

In [84]:
quote_distr = []
for i in range(len(lines_cleaned)):
    quote_distr.append([i, lines_cleaned[i]["quotes"]])

In [85]:
quote_distr

[[0, 0],
 [1, 0],
 [2, 1],
 [3, 0],
 [4, 0],
 [5, 1],
 [6, 0],
 [7, 0],
 [8, 0],
 [9, 0],
 [10, 0],
 [11, 0],
 [12, 0],
 [13, 0],
 [14, 0],
 [15, 0],
 [16, 0],
 [17, 0],
 [18, 0],
 [19, 0],
 [20, 0],
 [21, 0],
 [22, 0],
 [23, 0],
 [24, 0],
 [25, 0],
 [26, 0],
 [27, 0],
 [28, 0],
 [29, 0],
 [30, 0],
 [31, 0],
 [32, 0],
 [33, 0],
 [34, 0],
 [35, 0],
 [36, 0],
 [37, 0],
 [38, 0],
 [39, 1],
 [40, 0],
 [41, 0],
 [42, 0],
 [43, 1],
 [44, 0],
 [45, 0],
 [46, 0],
 [47, 0],
 [48, 0],
 [49, 0],
 [50, 0],
 [51, 0],
 [52, 0],
 [53, 0],
 [54, 0],
 [55, 0],
 [56, 0],
 [57, 0],
 [58, 0],
 [59, 0],
 [60, 0],
 [61, 0],
 [62, 0],
 [63, 0],
 [64, 0],
 [65, 0],
 [66, 0],
 [67, 0],
 [68, 0],
 [69, 0],
 [70, 0],
 [71, 0],
 [72, 0],
 [73, 0],
 [74, 0],
 [75, 0],
 [76, 0],
 [77, 0],
 [78, 0],
 [79, 0],
 [80, 0],
 [81, 0],
 [82, 0],
 [83, 0],
 [84, 0],
 [85, 0],
 [86, 0],
 [87, 0],
 [88, 0],
 [89, 0],
 [90, 0],
 [91, 0],
 [92, 1],
 [93, 0],
 [94, 0],
 [95, 0],
 [96, 0],
 [97, 0],
 [98, 0],
 [99, 0],
 [100, 0],

In [86]:
x_values = [item[0] for item in quote_distr]
y_values = [item[1] for item in quote_distr]

# Create bar plot
fig = px.bar(x=x_values, y=y_values, labels={'x': 'Liedvers', 'y': 'zitierende Predigten'}, title="Zitate aus „Wie schön leuchtet der Morgenstern“ je Liedvers")
fig.show()

In [23]:
# when do quote types spike?

sermon = Sermon("E000036")

In [57]:
def year_helper(year):
    year_finder = re.compile(r'[0-9]{4}')
    year_cleaned = re.findall(year_finder, year)[0]
    return int(year_cleaned)

In [101]:
quotes_per_year = []
sermons_per_year = []
for id in ids:
    sermon_quotes = 0
    source_quotes = 0
    song_quotes = 0

    sermon = Sermon(id)

    year = year_helper(sermon.erscheinungsjahr)
    link = f'<a href="https://orgelpredigt.ur.de/{id}" target="_blank">{sermon.kurztitel}</a>'
    sermons_per_year.append([year, sermon.kurztitel, link])

    all_quotes = set(sermon.all_references)
    for quote in all_quotes:
        if quote.startswith("E00"):
            sermon_quotes += 1
        elif quote.startswith("E08") or quote.startswith("E09"):
            source_quotes += 1
        elif quote.startswith("E10"):
            song_quotes += 1
    quotes_per_year.append([year, [sermon_quotes, source_quotes, song_quotes]])

In [103]:
quotes_per_year

[[1602, [0, 7, 0]],
 [1605, [0, 1, 0]],
 [1606, [0, 8, 5]],
 [1610, [0, 3, 1]],
 [1621, [0, 13, 1]],
 [1624, [0, 23, 0]],
 [1628, [0, 6, 1]],
 [1647, [1, 20, 8]],
 [1648, [2, 11, 8]],
 [1651, [1, 24, 7]],
 [1652, [2, 15, 5]],
 [1660, [1, 10, 1]],
 [1664, [0, 7, 4]],
 [1666, [0, 11, 3]],
 [1667, [0, 10, 8]],
 [1671, [0, 15, 4]],
 [1672, [0, 5, 12]],
 [1673, [3, 10, 4]],
 [1675, [3, 6, 5]],
 [1676, [2, 12, 13]],
 [1676, [0, 14, 8]],
 [1680, [3, 10, 7]],
 [1681, [0, 15, 18]],
 [1683, [1, 10, 16]],
 [1685, [2, 12, 1]],
 [1686, [0, 32, 7]],
 [1687, [1, 45, 1]],
 [1689, [0, 24, 1]],
 [1695, [1, 23, 3]],
 [1696, [1, 10, 0]],
 [1700, [0, 13, 10]],
 [1704, [0, 28, 3]],
 [1704, [2, 49, 3]],
 [1709, [1, 7, 1]],
 [1709, [0, 15, 5]],
 [1711, [2, 49, 15]],
 [1711, [0, 15, 2]],
 [1720, [2, 104, 5]],
 [1721, [0, 13, 0]],
 [1721, [3, 12, 11]],
 [1721, [2, 1, 1]],
 [1726, [0, 4, 5]],
 [1727, [0, 8, 6]],
 [1728, [0, 157, 8]],
 [1730, [0, 5, 6]],
 [1735, [0, 11, 17]],
 [1737, [0, 1, 6]],
 [1739, [2, 20, 3

In [93]:
sermons_per_year

[[1602,
  'Christliche Predigt (Tübingen 1602)',
  '<a href="https://orgelpredigt.ur.de/E000001" target="_blank">Christliche Predigt (Tübingen 1602)</a>'],
 [1605,
  'Musica instrumentalis (Meißen 1605)',
  '<a href="https://orgelpredigt.ur.de/E000002" target="_blank">Musica instrumentalis (Meißen 1605)</a>'],
 [1606,
  'Christliche Predigt (Tübingen 1606)',
  '<a href="https://orgelpredigt.ur.de/E000029" target="_blank">Christliche Predigt (Tübingen 1606)</a>'],
 [1610,
  'Elogium Organi Musici (Altenburg 1610)',
  '<a href="https://orgelpredigt.ur.de/E000030" target="_blank">Elogium Organi Musici (Altenburg 1610)</a>'],
 [1621,
  'Corona Templi (Nürnberg 1621)',
  '<a href="https://orgelpredigt.ur.de/E000099" target="_blank">Corona Templi (Nürnberg 1621)</a>'],
 [1624,
  'Vlmische Orgel Predigt (Ulm 1624)',
  '<a href="https://orgelpredigt.ur.de/E000003" target="_blank">Vlmische Orgel Predigt (Ulm 1624)</a>'],
 [1628,
  'Musica ecclesiastica (Stettin 1628)',
  '<a href="https://orgel

In [94]:
colors = {
    'Orgelpredigten': 'rgb(135, 44, 162)',
    'Musikwerke': 'rgb(192, 54, 157)',
    'Literatur': 'rgb(234, 79, 136)'
}

In [95]:
df = pd.DataFrame(sermons_per_year, columns=["year", "title", "id"])
# Create full year range
all_years = pd.DataFrame({"year": range(1600,1801)})

# Merge to ensure every year appears, fill missing with 0
df_full = pd.merge(all_years, df, on="year", how="left").fillna(0)

agg = df.groupby("year").agg({
    "title": lambda x: "<br>".join(x.astype(str)),
    "id": lambda x: ", ".join(x.astype(str)),
    "year": "count"
}).rename(columns={"year": "count"}).reset_index()

In [99]:
fig = px.bar(agg, x='year', y='count',
             hover_data='id',
             color_discrete_sequence=['rgb(135, 44, 162)'])
fig.show()

In [102]:
df = pd.DataFrame(quotes_per_year, columns=["Jahr", "Anzahl"])
df[["Orgelpredigten", "Literatur", "Musikwerke"]] = pd.DataFrame(df["Anzahl"].tolist(), index=df.index)
df = df.drop(columns="Anzahl")

# Melt to long format for plotly express
df_long = df.melt(id_vars="Jahr", value_vars=["Orgelpredigten", "Literatur", "Musikwerke"],
                  var_name="type", value_name="Anzahl")

# Create figure
fig = go.Figure()

# Add line traces with custom colors
for t in df_long["type"].unique():
    subset = df_long[df_long["type"] == t]
    fig.add_trace(
        go.Scatter(
            x=subset["Jahr"],
            y=subset["Anzahl"],
            mode="lines+markers",
            name=t,
            line=dict(color=colors[t], width=2),
            marker=dict(size=8)
        )
    )

# Layout
fig.update_layout(
    title="Zeitliche Verteilung von Zitattypen",
    xaxis_title="Jahr",
    yaxis_title="Anzahl"
)

fig.show()

In [104]:
sermon = Sermon("E000036")

In [122]:
sermon.chunked[0][3]["types"]

['', '', '', '', '', ' bibel', ' bibel']

In [ ]:
t = ["", "", ""]

In [119]:
first_type = ""
for i in t:
    if (len(i) != 0):
        first_type = i
        break
print(first_type)

 .
